# Задача 3. Афинные штрафы за гэпы

Нужно реализовать алгоритм Нидлмана-Вунша на Python и рассчитать Score для выравнивания:
- **Seq1:** `ATGCAGCAGCAGCCA`
- **Seq2:** `ATATAT`

При двух моделях:
1. **Линейный штраф:** Gap = -4
2. **Аффинный штраф:** Open = -10, Extend = -1

In [11]:
import numpy as np

In [12]:
import numpy as np

def needleman_wunsch_linear(s1, s2, match=1, mismatch=-1, gap=-4):
    n, m = len(s1), len(s2)
    F = np.zeros((n+1, m+1), dtype=int)
    traceback = np.zeros((n+1, m+1), dtype=int)


    #инициализация
    for i in range(1, n+1):
        F[i,0] = F[i-1,0] + gap
        traceback[i,0] = 1

    for j in range(1, m+1):
        F[0,j] = F[0,j-1] + gap
        traceback[0,j] = 2

    #заполнение
    for i in range(1, n+1):
        for j in range(1, m+1):
            score = match if s1[i-1] == s2[j-1] else mismatch

            diag = F[i-1,j-1] + score
            up   = F[i-1,j] + gap
            left = F[i,j-1] + gap

            F[i,j] = max(diag, up, left)

            if F[i,j] == diag:
                traceback[i,j] = 0
            elif F[i,j] == up:
                traceback[i,j] = 1
            else:
                traceback[i,j] = 2

    #обратный ход
    align1 = []
    align2 = []
    i, j = n, m

    while i > 0 or j > 0:
        if traceback[i,j] == 0:
            align1.append(s1[i-1])
            align2.append(s2[j-1])
            i -= 1
            j -= 1
        elif traceback[i,j] == 1:
            align1.append(s1[i-1])
            align2.append('-')
            i -= 1
        else:
            align1.append('-')
            align2.append(s2[j-1])
            j -= 1

    align1 = ''.join(reversed(align1))
    align2 = ''.join(reversed(align2))

    print("Выравнивание с линейным штрафом")
    print(F)
    print("Score:", F[n,m])
    print("S1:", align1)
    print("S2:", align2)
    print()

    return F[n,m], align1, align2

**Афинный штраф**

In [13]:
def needleman_wunsch_affine(s1, s2, match=1, mismatch=-1, gap_open=-10, gap_extend=-1):

    n, m = len(s1), len(s2)

    NEG_INF = -10**9

    M  = np.full((n+1, m+1), NEG_INF)
    Ix = np.full((n+1, m+1), NEG_INF)
    Iy = np.full((n+1, m+1), NEG_INF)
    
    #матрицы обратного хода - будем заполнять откуда прили для того или иного шага 
    tbM  = np.zeros((n+1, m+1), dtype=int)
    tbIx = np.zeros((n+1, m+1), dtype=int)
    tbIy = np.zeros((n+1, m+1), dtype=int)

    M[0,0] = 0

    #инициализация
    for i in range(1, n+1):
        Ix[i,0] = gap_open + (i-1)*gap_extend
        tbIx[i,0] = 1

    for j in range(1, m+1):
        Iy[0,j] = gap_open + (j-1)*gap_extend
        tbIy[0,j] = 2

    #заполнение
    for i in range(1, n+1):
        for j in range(1, m+1):
            score = match if s1[i-1] == s2[j-1] else mismatch

            # M
            choices = [M[i-1,j-1], Ix[i-1,j-1], Iy[i-1,j-1]]
            M[i,j] = max(choices) + score
            tbM[i,j] = choices.index(max(choices))

            # Ix
            open_gap  = M[i-1,j] + gap_open
            extend_gap = Ix[i-1,j] + gap_extend
            Ix[i,j] = max(open_gap, extend_gap)
            tbIx[i,j] = 0 if open_gap >= extend_gap else 1

            # Iy
            open_gap  = M[i,j-1] + gap_open
            extend_gap = Iy[i,j-1] + gap_extend
            Iy[i,j] = max(open_gap, extend_gap)
            tbIy[i,j] = 0 if open_gap >= extend_gap else 1

    #обратный ход - чтобы вывести выравнивание
    i, j = n, m
    matrices = [M[i,j], Ix[i,j], Iy[i,j]]
    state = matrices.index(max(matrices))  # 0=M, 1=Ix, 2=Iy

    align1 = []
    align2 = []

    while i > 0 or j > 0:
        if state == 0:  # M
            prev_state = tbM[i,j]
            align1.append(s1[i-1])
            align2.append(s2[j-1])
            i -= 1
            j -= 1
            state = prev_state

        elif state == 1:  # Ix 
            prev_state = tbIx[i,j]
            align1.append(s1[i-1])
            align2.append('-')
            i -= 1
            state = prev_state

        else:  # Iy 
            prev_state = tbIy[i,j]
            align1.append('-')
            align2.append(s2[j-1])
            j -= 1
            state = prev_state

    align1 = ''.join(reversed(align1))
    align2 = ''.join(reversed(align2))

    print("Выравнивание с афинным штрафом")
    print("Матрица M")
    print(M)
    print("Матрица Ix")
    print(Ix)
    print("Матрица Iy")
    print(Iy)
    print("Score:", max(matrices))
    print("S1:", align1)
    print("S2:", align2)

In [14]:
s1 = "ATGCAGCAGCAGCCA"
s2 = "ATATAT"

needleman_wunsch_linear(s1, s2)
needleman_wunsch_affine(s1, s2)

Выравнивание с линейным штрафом
[[  0  -4  -8 -12 -16 -20 -24]
 [ -4   1  -3  -7 -11 -15 -19]
 [ -8  -3   2  -2  -6 -10 -14]
 [-12  -7  -2   1  -3  -7 -11]
 [-16 -11  -6  -3   0  -4  -8]
 [-20 -15 -10  -5  -4   1  -3]
 [-24 -19 -14  -9  -6  -3   0]
 [-28 -23 -18 -13 -10  -7  -4]
 [-32 -27 -22 -17 -14  -9  -8]
 [-36 -31 -26 -21 -18 -13 -10]
 [-40 -35 -30 -25 -22 -17 -14]
 [-44 -39 -34 -29 -26 -21 -18]
 [-48 -43 -38 -33 -30 -25 -22]
 [-52 -47 -42 -37 -34 -29 -26]
 [-56 -51 -46 -41 -38 -33 -30]
 [-60 -55 -50 -45 -42 -37 -34]]
Score: -34
S1: ATGCAGCAGCAGCCA
S2: AT-----A-TA---T

Выравнивание с афинным штрафом
Матрица M
[[          0 -1000000000 -1000000000 -1000000000 -1000000000 -1000000000
  -1000000000]
 [-1000000000           1         -11         -10         -13         -12
          -15]
 [-1000000000         -11           2         -10          -9         -12
          -11]
 [-1000000000         -12         -10           1          -9         -10
          -11]
 [-1000000000         

**Какую биологическую особенность лучше описывает аффинная модель**

Афинная модель показывает, что повяление разрывов (или вставка) проосиходит блоками - то есть если гэп появился - скорее всего это петля в белке и её длина не сильно влияет на его физическо-химиечские свойства, и следовательно мы меньше штрафуем за удлинение этого гэпа. (еще на лекции был пример на афинные штрафы, что если мы выравниваем нуклеотидные последовательности: один гэп - это сдвиг рамки считывания, поэтому лучше три вместе - удлиненный гэп - чем три по одному). 